# CAP4D Static Avatar for Unity (Colab)
This notebook runs tracking + generation + avatar fitting and exports a static Unity-friendly 3DGS `.ply`.

In [ ]:
# Top-level settings
QUALITY = "balanced"   # "balanced" | "max" | "debug"
MAX_N_REF = 48           # reduce for speed, increase for quality
TIMESTEP = 0             # static FLAME timestep to bake
INPUT_VIDEO_PATH = "/content/my_head_video.mp4"
OUTPUT_PATH = "/content/cap4d/examples/output/custom_static"
REPO_URL = "https://github.com/vikram-menon/cap4d.git"  # set to your fork if needed
REPO_REF = "colab"  # branch/tag/commit containing static export scripts

In [ ]:
!nvidia-smi -L

In [ ]:
import subprocess
%cd /content
!rm -rf cap4d
subprocess.run(["git", "clone", REPO_URL, "cap4d"], check=True)
%cd /content/cap4d
subprocess.run(["git", "checkout", REPO_REF], check=True)

In [ ]:
import os
os.environ["CAP4D_PATH"] = "/content/cap4d"
os.environ["PIXEL3DMM_PATH"] = "/content/pixel3dmm"
os.environ["PYTHONPATH"] = f"/content/cap4d:{os.environ.get('PYTHONPATH', '')}"
print('CAP4D_PATH=', os.environ['CAP4D_PATH'])
print('PIXEL3DMM_PATH=', os.environ['PIXEL3DMM_PATH'])

In [ ]:
%%bash
set -e
cd /content/cap4d
pip install -r requirements.txt
export FORCE_CUDA=1
pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"
apt-get update
apt-get install -y ffmpeg

In [ ]:
import os
import getpass
os.environ['FLAME_USERNAME'] = input('FLAME username: ')
os.environ['FLAME_PWD'] = getpass.getpass('FLAME password: ')

In [ ]:
%%bash
set -e
cd /content/cap4d
bash scripts/download_flame.sh
bash scripts/download_mmdm_weights.sh
bash scripts/install_pixel3dmm.sh

In [ ]:
# Optional upload: use this cell if INPUT_VIDEO_PATH is not already available in /content
from google.colab import files
uploaded = files.upload()
if uploaded:
    INPUT_VIDEO_PATH = f"/content/{next(iter(uploaded.keys()))}"
print('INPUT_VIDEO_PATH =', INPUT_VIDEO_PATH)

In [ ]:
import subprocess
cmd = [
    'bash', 'scripts/generate_static_avatar.sh',
    INPUT_VIDEO_PATH, OUTPUT_PATH, QUALITY,
    '--max_n_ref', str(MAX_N_REF),
    '--timestep', str(TIMESTEP),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd='/content/cap4d', check=True)

In [ ]:
from google.colab import files
ply_path = f"{OUTPUT_PATH}/raw_static.ply"
print('Exported PLY:', ply_path)
files.download(ply_path)